# 03 - Feature Analysis

Explores `outputs/feature_analysis/feature_registry.csv` (the auto-generated canonical feature list, built by `scripts/build_modeling_dataset.py`) alongside redundancy/multicollinearity checks on the modeling table.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import pandas as pd
from cfb_win_total_model.dataset import NON_FEATURE_COLS
from cfb_win_total_model.utils.paths import DATA_PROCESSED_DIR, OUTPUTS_FEATURE_ANALYSIS

registry = pd.read_csv(OUTPUTS_FEATURE_ANALYSIS / "feature_registry.csv")
df = pd.read_parquet(DATA_PROCESSED_DIR / "modeling_dataset.parquet")
registry.shape, df.shape

In [ ]:
registry["category"].value_counts()

In [ ]:
# Cross-check: registry and dataset columns must match exactly (also enforced by tests/test_modeling_dataset.py)
reg_names = set(registry["feature_name"])
df_cols = set(df.columns) - NON_FEATURE_COLS
print("in df but not registry:", sorted(df_cols - reg_names))
print("in registry but not df:", sorted(reg_names - df_cols))

In [ ]:
# Highly correlated feature pairs (redundancy check)
numeric_cols = df[list(reg_names)].select_dtypes(include="number").columns
corr = df[numeric_cols].corr().abs()
high_corr = (
    corr.stack()
    .reset_index()
    .rename(columns={"level_0": "feature_a", "level_1": "feature_b", 0: "abs_corr"})
)
high_corr = high_corr[(high_corr["feature_a"] < high_corr["feature_b"]) & (high_corr["abs_corr"] > 0.9)]
high_corr.sort_values("abs_corr", ascending=False).head(30)

In [ ]:
# Permutation / feature importance from the final holdout, if evaluate_models.py has run
from cfb_win_total_model.utils.paths import OUTPUTS_DIAGNOSTICS
perm_path = OUTPUTS_DIAGNOSTICS / "permutation_importance.csv"
if perm_path.exists():
    display(pd.read_csv(perm_path).head(20))
else:
    print("Run scripts/evaluate_models.py first")